# Notebook 03 – Matrixsteifigkeitsmethode: Aufruf über Funktionen

In Notebook 01 und 02 haben wir die gesamte FE-Logik schrittweise erarbeitet und in Funktionen gekapselt.
Diese Funktionen sind nun in **`fem_core.py`** ausgelagert:

| Funktion | Aufgabe |
|---|---|
| `assemble_K(...)` | Assembliert die SSM $\underline{\underline{K}}$ |
| `solve_system(K, constraints, loads)` | Löst das reduzierte LGS, gibt $\boldsymbol{U}$ und $\boldsymbol{F}$ zurück |
| `plot_results(...)` | Visualisiert verformte und unverformte Konfiguration |

**Workflow:** Modell definieren → `assemble_K` → `solve_system` → `postprocessing` → `plot_results`.

In [ ]:
import os, sys
if not os.path.exists("FEM"):
    !git clone --depth=1 -q https://github.com/Boscij/FEM.git
if "FEM/content" not in sys.path:
    sys.path.insert(0, "FEM/content")

In [ ]:
import numpy as np
from fem_core import assemble_K, solve_system
from fem_post import postprocessing, plot_results

## Modell

Koordinaten in $[\text{mm}]$, Querschnittsflächen in $[\text{mm}^2]$, $E$ in $[\text{MPa} = \text{N/mm}^2]$, Kräfte in $[\text{N}]$.

In [ ]:
# Knotenkoordinaten [X, Y] in mm  (Knoten 1..4 → Index 0..3)
nodal_coordinates = np.array([
    [   0.0,    0.0],   # Knoten 1
    [1000.0,    0.0],   # Knoten 2
    [1000.0, 1000.0],   # Knoten 3
    [2000.0, 1000.0],   # Knoten 4
])

# Elemente: [Knoten_i, Knoten_j, section_key]  (0-basiert; Stab 1..5 → Index 0..4)
elements = [
    [0, 1, "section 1"],   # Stab 1: Knoten 1–2
    [0, 2, "section 2"],   # Stab 2: Knoten 1–3
    [1, 2, "section 3"],   # Stab 3: Knoten 2–3
    [1, 3, "section 4"],   # Stab 4: Knoten 2–4
    [2, 3, "section 5"],   # Stab 5: Knoten 3–4
]

# Materialien: E-Modul [MPa = N/mm²]
materials = {"steel": [210000.0]}

# Querschnitte: [A [mm²], material_key]
sections = {
    "section 1": [15.00, "steel"],
    "section 2": [28.28, "steel"],
    "section 3": [10.00, "steel"],
    "section 4": [56.56, "steel"],
    "section 5": [10.00, "steel"],
}

# Randbedingungen: [Knoten (0-basiert), Achse (0=x, 1=y), vorgegebene Verschiebung]
constraints = [
    [0, 0, 0.0],   # Knoten 1: U_1 = 0  (Festlager, x)
    [0, 1, 0.0],   # Knoten 1: U_2 = 0  (Festlager, y)
    [1, 1, 0.0],   # Knoten 2: U_4 = 0  (Loslager, y)
]

# Lasten: [Knoten (0-basiert), Achse (0=x, 1=y), Kraft [N]]
loads = [
    [3, 1, -1000.0],   # Knoten 4: F_y = -1000 N
]

## Lösung

Das gesamte FE-Problem lässt sich mit **zwei Funktionsaufrufen** lösen.

In [ ]:
K        = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed = solve_system(K, constraints, loads)

np.set_printoptions(precision=6, suppress=True)
print("Verschiebungen U [mm]:")
for k, u in enumerate(U):
    mark = "  ← gesperrt" if fixed[k] else ""
    print(f"  U_{k+1} = {u:+.6e} mm{mark}")

print("\nKräfte F [N] (inkl. Reaktionen):")
for k, f in enumerate(F):
    mark = "  ← Reaktion" if fixed[k] else ""
    print(f"  F_{k+1} = {f:+.4f} N{mark}")

## Post-Processing

### Spannungen und Dehnungen

Die globalen Knotenverschiebungen des Elements werden zunächst auf die lokale Stabachse projiziert:

$$
\boldsymbol{u}^e = \underline{\underline{T}}\,\boldsymbol{U}^e, \qquad
\boldsymbol{U}^e = \begin{bmatrix} U^e_1 \\ U^e_2 \\ U^e_3 \\ U^e_4 \end{bmatrix}, \qquad
\boldsymbol{u}^e = \begin{bmatrix} u^e_1 \\ u^e_2 \end{bmatrix}, \qquad
\underline{\underline{T}} = \begin{bmatrix} c & s & 0 & 0 \\ 0 & 0 & c & s \end{bmatrix}
$$

Daraus folgen Dehnung, Spannung und Normalkraft:

$$
\varepsilon_e = \frac{u^e_2 - u^e_1}{L}, \qquad
\sigma_e = E\,\varepsilon_e, \qquad
N_e = \sigma_e \cdot A
$$

In [ ]:
eps, sig, N = postprocessing(U, nodal_coordinates, elements, sections, materials)

print(f"{'Stab':>4}  {'ε [-]':>14}  {'σ [MPa]':>10}  {'N [N]':>10}  Zustand")
print("─" * 58)
for e in range(len(elements)):
    z = "Zug" if N[e] > 0 else ("Druck" if N[e] < 0 else "0")
    print(f"  {e+1:>2}  {eps[e]:>14.6e}  {sig[e]:>10.4f}  {N[e]:>10.4f}  {z}")

### Visualisierung

Unverformte Referenzkonfiguration (grau gestrichelt) und verformte Konfiguration eingefärbt nach Spannung $\sigma$ [MPa].

In [ ]:
plot_results(nodal_coordinates, elements, constraints, loads, U, sig, scale=200)